# MTHFVRP 模型 (Model) 架构与使用指南

本 Notebook 详细解析了 `TransformerModel` 的网络架构、输入特征流向，以及如何与 `MTHFVRPEnv` 环境协同工作。

主要内容：
1. **输入特征详解**：结合环境特征提取，说明模型的输入数据流。
2. **模型架构图解**：展示 Encoder-Decoder 结构及 Attention 机制。
3. **交互流程演示**：代码示例展示 “环境 -> 模型 -> 动作 -> 环境” 的闭环交互。

In [1]:
%load_ext autoreload
%autoreload 2

import sys
import os
import torch
from torch import nn
from tensordict import TensorDict

# 将项目根目录添加到 Python 路径
project_root = os.path.dirname(os.getcwd())
if project_root not in sys.path:
    print(f"Adding project root to Python path: {project_root}")
    sys.path.insert(0, project_root)

# 导入必要模块
from implement.generator import MTHFVRPGenerator
from implement.environment import MTHFVRPEnv
from implement.model import TransformerModel

Adding project root to Python path: /home/huangsh/SF_NCO/mvhfvrp_20260204


## 1. 输入特征详解 (Input Features)

模型主要接收两类特征：`Global Features` (用于 Encoder) 和 `Current Features` (用于 Decoder)。

### 1.1 全局特征 (Global Features - for Encoder)
这些特征在整个 Episode 开始时提取一次，并通过 Encoder 编码为 Embeddings。

在 `environment.py` 的 `get_global_features` 中生成，包含：

1.  **Problem Features (问题属性)**
    *   **来源**: `state_feature -> problem_feature`
    *   **维度**: `[B, D_problem]`
    *   **内容**: Capacity, Open, TW, Limit, Backhaul, HF 等全局标志位。
    *   **用途**: 初始化 Problem Feature Embedding。

2.  **Depot & Node Features (节点属性)**
    *   **来源**: `state_feature -> depot_features` & `node_features`
    *   **维度**: `[B, 1+N, D_node]`
    *   **内容**: 坐标、需求 (Linehaul/Backhaul)、时间窗、服务时间。
    *   **用途**: 通过 `ProblemNet` 和 `NodeSelfAttentionEncoder` 编码为 `node_embeddings`。

3.  **Vehicle Features (车辆属性)**
    *   **来源**: `state_feature -> vehicle_features`
    *   **维度**: `[B, V, D_vehicle]`
    *   **内容**: 车辆容量、固定成本、可变成本、可用数量。
    *   **用途**: 通过 `VehicleSelfAttentionEncoder` 编码为 `vehicle_embeddings`。

### 1.2 当前决策特征 (Current Features - for Decoder)
这些特征在每个 Step 中动态生成，用于 Decoder 生成当前步的策略。

在 `environment.py` 的 `get_current_feature_and_mask` 中生成，包含：

1.  **Context Features (上下文特征)**
    *   **来源**: `current_features` 返回值
    *   **维度**: `[B, D_curr]`
    *   **内容**:
        *   当前节点索引
        *   当前车辆类型
        *   当前车辆剩余数量
        *   当前剩余容量 (Linehaul/Backhaul)
        *   当前累积时间、距离
    *   **用途**: 作为 Decoder 的 Query (`q`) 的一部分，提供此时此刻的状态信息。

2.  **Illegal Mask (动作掩码)**
    *   **来源**: `illegal_mask` 返回值
    *   **维度**: `[B, 1+V+N]`
    *   **用途**: 在 Decoder 的 Attention 计算中屏蔽不可行节点 (`-inf` masking)。

## 2. 模型架构框图 (Model Framework)

`TransformerModel` 采用了 **Encoder-Decoder** 架构，并针对 VRP 问题的异构节点（Depot, Customers, Vehicles）设计了多流注意力机制。

### 2.1 架构简图

```mermaid
graph TD
    subgraph Inputs
        Global[Global Features\n(Problem, Node, Vehicle)]
        Current[Current Features\n(Step Context)]
    end

    subgraph "Encoder (Pre-computation)"
        ProbNet[ProblemNet] --> ProbEmb[Problem Emb]
        
        Global --> NodeEnc[Node Self-Attention] --> NodeEmb[Node Embeddings]
        Global --> VehEnc[Vehicle Self-Attention] --> VehEmb[Vehicle Embeddings]
        
        NodeEmb & VehEmb --> CrossEnc[Vehicle-Node Cross-Attention] --> VehNodeEmb[Veh-Node Embeddings]
        
        ProbEmb & VehNodeEmb & NodeEmb --> GlobalEnc[Global Node Encoder] --> FinalGlobalEmb[Final Global Embeddings]
    end

    subgraph "Decoder (Per Step)"
        Current --> DecQuery[Feature Embedding -> Query]
        
        FinalGlobalEmb --> KV_Global[KV Cache: Global]
        NodeEmb --> KV_Node[KV Cache: Node]
        VehEmb --> KV_Vehicle[KV Cache: Vehicle]
        
        DecQuery & KV_Global & KV_Node & KV_Vehicle --> PointerAttn[Multi-Head Pointer Attention]
        
        PointerAttn --> Logits[Action Logits]
    end

    Inputs --> Encoder
    Encoder --> Decoder
    Decoder --> Actions
```

### 2.2 核心模块描述

1.  **Feature Encoder (Initial Embedding)**:
    *   使用 `ProblemNet` 和 `nn.Linear` 将原始特征投影到高维隐空间 `hidden_size`。

2.  **Structural Encoder (Attention Stack)**:
    *   **Node Self-Attention**: 处理客户节点间的拓扑关系。
    *   **Vehicle Self-Attention**: 处理异构车辆间的关系。
    *   **Vehicle-Node Cross-Attention**: 融合车辆能力与客户需求信息。
    *   **Global Node Encoder**: 将问题Embedding、车辆Embedding和节点Embedding进行深度融合，生成最终的 `Global Embeddings`。

3.  **Pointer Decoder (Policy Head)**:
    *   这是一个轻量级的 Decoder。
    *   **Query**: 由当前的步进特征 (`Context Features`) 生成。
    *   **Key/Value**: 来自 Encoder 预计算好的各种 Embeddings (Global, Node, Vehicle)。这使得 Decoder 极其高效，不需要在每一步重新计算整个图的 Embedding。
    *   **Attention**: 计算 Query 与所有候选动作 (Depot, Vehicles, Customers) 的相似度，经过 Softmax 得到概率分布。
    *   **Masking**: 应用 `illegal_mask` 确保只采样合法动作。

## 3. 模型与环境交互演示

下面的代码演示了完整的闭环：
1.  **初始化**: 创建环境和模型。
2.  **预编码**: 调用 `model.feature()` 一次性计算所有静态 Embeddings。
3.  **循环交互**:
    *   `env` 提供当前特征 -> `model.policy()` 输出动作概率。
    *   采样动作 -> `env.step()` 更新状态。

In [2]:
def demo_model_env_interaction():
    # 1. 准备配置
    NODE_NUM = 20
    VEHICLE_NUM = 5
    BATCH_SIZE = 2  # 演示 Batch 操作
    
    device = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"Using device: {device}")

    # 2. 初始化生成器和环境
    # disable check_solution for speed in demo
    generator = MTHFVRPGenerator(num_loc=NODE_NUM, vehicle_num=VEHICLE_NUM, variant_preset="hfcvrp")
    env = MTHFVRPEnv(generator=generator, batch_size=[BATCH_SIZE], device=device)

    # 3. 初始化模型
    # 我们需要先跑一次 reset 获取特征维度，以便初始化模型参数
    tmp_state = env.reset()
    input_features = env.get_global_features(tmp_state)
    
    # 自动推断特征维度
    state_feature_dims = {}
    for key, feature in input_features["state_feature"].items():
        state_feature_dims[key] = feature.shape[-1]
    
    # 获取 current feature 维度 (减2是因为模型内部会把 current node/vehicle idx 拿去做 embedding，不算在数值特征里)
    tmp_curr_feat, _ = env.get_current_feature_and_mask(tmp_state)
    state_feature_dims['current_feature'] = tmp_curr_feat.shape[-1] - 2 
    
    print("\n特征维度推断:", state_feature_dims)

    # 初始化 Transformer 模型
    model = TransformerModel(
        hidden_size=128,
        n_head=8,
        encoder_num_layers=3, # 演示用，层数少点
        state_feature_dims=state_feature_dims
    ).to(device)
    
    model.eval() # 推理模式

    print("\n=== 开始交互流程 ===")
    
    # A. Reset 环境
    state = env.reset()
    
    # B. Model Encoding (只做一次!)
    # 获取全局特征
    global_features_td = env.get_global_features(state)
    # 编码，此时模型内部会缓存 KV Cache (self.cache_*)
    model.feature(global_features_td)
    print("Encoder 编码完成，KV Cache 已准备就绪。")
    
    # C. Rollout Loop
    done = False
    step = 0
    total_rewards = torch.zeros(BATCH_SIZE, device=device)
    
    while not done:
        # 1. 获取当前步的 Context (B, D_curr) 和 Mask (B, ActionSpace)
        current_features, illegal_mask = env.get_current_feature_and_mask(state)
        
        # 2. Model Decoding
        # 输入: 当前特征 + Mask
        # 输出: 动作 Logits (未归一化的概率)
        with torch.no_grad():
            action_logits = model.policy(current_features, illegal_mask)
        
        # 3. 动作选择 (Sampling)
        action_probs = torch.softmax(action_logits, dim=-1)
        dist = torch.distributions.Categorical(action_probs)
        action = dist.sample()
        
        # 4. Environment Step
        state, reward, is_done_tensor = env.step(state, action)
        
        # 累加奖励 (注意 masked reward)
        total_rewards += reward.squeeze(-1)
        
        done = is_done_tensor.all().item()
        step += 1
        
        # 打印部分动作 (只看 Batch 0)
        if step <= 10: # 防止刷屏
            print(f"Step {step}: User 0 Action {action[0].item()} | Reward {reward[0].item():.2f}")
            
    print(f"\nEpisode 结束，共 {step} 步。")
    print(f"Final Rewards: {total_rewards.cpu().numpy()}")

# 运行演示
demo_model_env_interaction()

Using device: cuda

特征维度推断: {'problem_feature': 6, 'depot_features': 4, 'node_features': 7, 'vehicle_features': 4, 'current_feature': 5}

=== 开始交互流程 ===
Encoder 编码完成，KV Cache 已准备就绪。
Step 1: User 0 Action 3 | Reward -0.06
Step 2: User 0 Action 20 | Reward -0.18
Step 3: User 0 Action 0 | Reward -0.18
Step 4: User 0 Action 4 | Reward -0.06
Step 5: User 0 Action 24 | Reward -0.13
Step 6: User 0 Action 13 | Reward -0.27
Step 7: User 0 Action 21 | Reward -0.17
Step 8: User 0 Action 8 | Reward -0.13
Step 9: User 0 Action 9 | Reward -0.22
Step 10: User 0 Action 0 | Reward -0.24

Episode 结束，共 28 步。
Final Rewards: [-14.500986 -10.030357]
